# 09 · Infrastructure monitoring: alarms, SNS and the CloudWatch dashboard
**AAI-540 · Group 4 · Criteo CTR**

The model monitors in notebook 08 answer "is the model still right". This
notebook answers the question an operator asks first, at 3am: **is it serving
at all, is it fast enough, and is the host healthy** — and makes the answer one
dashboard rather than six consoles.

Three things here are deliberate rather than conventional:

* **Alarm definitions are data, not inline calls.** `monitoring.alarm_specs`
  returns a list of `put_metric_alarm` keyword arguments, which means the whole
  alarm set is reviewable in a pull request and asserted in
  `tests/test_monitoring.py`. An alarm created by hand in a console is an alarm
  nobody can diff.
* **Latency thresholds are in microseconds.** That is the unit CloudWatch uses
  for SageMaker latency metrics, and a threshold of `100` written in the belief
  it means milliseconds is a routine reason an alarm never fires.
* **`TreatMissingData="notBreaching"`.** This endpoint serves in bursts. An
  alarm that goes to ALARM every time traffic stops is an alarm that gets
  muted, and a muted alarm is worse than no alarm because it looks like
  coverage.

The dashboard puts model-quality metrics *beside* infrastructure metrics on one
screen, so a quality regression and a capacity problem are never mistaken for
each other — a slow endpoint and a drifting model need completely different
responses.

In [ ]:
import os, sys, json, time, datetime as dt
sys.path.insert(0, os.path.abspath("../src"))

import boto3
import numpy as np
import pandas as pd
import sagemaker

from criteo_ctr import config as C
from criteo_ctr import monitoring as M
from criteo_ctr import pipeline as P
from criteo_ctr.io_utils import Store, s3_uri

sess = sagemaker.Session()
boto_sess = sess.boto_session
region = sess.boto_region_name
bucket = C.BUCKET or sess.default_bucket()
store = Store(bucket, boto_sess)

cw = boto_sess.client("cloudwatch")
sns = boto_sess.client("sns")
sm = boto_sess.client("sagemaker")

ENDPOINT_NAME = "criteo-ctr-endpoint"
DASHBOARD_NAME = "criteo-ctr-ml-system"
ALERT_EMAIL = os.environ.get("CRITEO_ALERT_EMAIL", "")   # set to subscribe

FIG_DIR = os.path.abspath("../reports/figures"); os.makedirs(FIG_DIR, exist_ok=True)
print(f"region={region}\nendpoint={ENDPOINT_NAME}")
print(f"endpoint status: "
      f"{sm.describe_endpoint(EndpointName=ENDPOINT_NAME)['EndpointStatus']}")

## 1 · SNS topic for alerts

An alarm with no action is a dashboard decoration. The topic is created first so
every alarm below has somewhere to deliver.

Email subscriptions only deliver **after the recipient confirms** — which held
up our alarm testing for a while and is worth knowing before a demo.

In [ ]:
topic_arn = sns.create_topic(Name="criteo-ctr-alerts")["TopicArn"]
print(f"topic: {topic_arn}")

if ALERT_EMAIL:
    sns.subscribe(TopicArn=topic_arn, Protocol="email", Endpoint=ALERT_EMAIL)
    print(f"subscription requested for {ALERT_EMAIL} — confirm the email before testing")
else:
    print("set CRITEO_ALERT_EMAIL to subscribe an address")

subscriptions = sns.list_subscriptions_by_topic(TopicArn=topic_arn)["Subscriptions"]
display(pd.DataFrame([{"protocol": s["Protocol"], "endpoint": s["Endpoint"],
                       "confirmed": s["SubscriptionArn"].startswith("arn:")}
                      for s in subscriptions]) if subscriptions
        else "no subscriptions yet")

## 2 · Create the alarms from the reviewed specification

In [ ]:
specs = M.alarm_specs(ENDPOINT_NAME, topic_arn)
display(pd.DataFrame([{"alarm": s["AlarmName"], "metric": s["MetricName"],
                       "namespace": s["Namespace"],
                       "statistic": s.get("Statistic") or s.get("ExtendedStatistic"),
                       "condition": f"{s['ComparisonOperator']} {s['Threshold']:g}",
                       "period_s": s["Period"], "periods": s["EvaluationPeriods"]}
                      for s in specs]))

for spec in specs:
    cw.put_metric_alarm(**spec)
print(f"\ncreated/updated {len(specs)} alarms")

In [ ]:
names = [s["AlarmName"] for s in specs]
described = cw.describe_alarms(AlarmNames=names)["MetricAlarms"]
display(pd.DataFrame([{"alarm": a["AlarmName"], "state": a["StateValue"],
                       "reason": a["StateReason"][:70],
                       "actions": len(a["AlarmActions"])}
                      for a in described]))

print("\nINSUFFICIENT_DATA on a freshly created alarm is expected: the metric has "
      "no\ndatapoints in the evaluation window yet. Invoke the endpoint (notebook 07) "
      "and\nre-run this cell to watch them move to OK.")

## 3 · Build the dashboard

One screen, read top to bottom in triage order: traffic → errors → latency →
host → model quality → batch and pipeline activity. `monitoring.dashboard_body`
builds the JSON so the layout is version-controlled and unit-tested rather than
dragged into place in a console and lost on the next account.

In [ ]:
body = M.dashboard_body(endpoint_name=ENDPOINT_NAME, region=region,
                        pipeline_name=P.PIPELINE_NAME)
cw.put_dashboard(DashboardName=DASHBOARD_NAME, DashboardBody=body)

parsed = json.loads(body)
print(f"dashboard '{DASHBOARD_NAME}' created with {len(parsed['widgets'])} widgets:")
for widget in parsed["widgets"]:
    title = widget["properties"].get("title", "(header)")
    metrics = len(widget["properties"].get("metrics", []))
    print(f"  {widget['type']:7s} {title:42s} {metrics} metric(s)")

console = (f"https://{region}.console.aws.amazon.com/cloudwatch/home"
           f"?region={region}#dashboards:name={DASHBOARD_NAME}")
print(f"\nopen: {console}")
store.put_text(body, f"{C.PREFIX}/monitoring/dashboard/{DASHBOARD_NAME}.json")

## 4 · Read the metrics back

Proof the dashboard is wired to real data rather than empty widgets — and the
numbers that go in the design document's deployment section.

In [ ]:
end = dt.datetime.utcnow()
start = end - dt.timedelta(hours=3)
dimensions = [{"Name": "EndpointName", "Value": ENDPOINT_NAME},
              {"Name": "VariantName", "Value": "AllTraffic"}]

queries = [
    ("invocations", "AWS/SageMaker", "Invocations", "Sum"),
    ("4xx errors", "AWS/SageMaker", "Invocation4XXErrors", "Sum"),
    ("5xx errors", "AWS/SageMaker", "Invocation5XXErrors", "Sum"),
    ("model latency p50 (us)", "AWS/SageMaker", "ModelLatency", "p50"),
    ("model latency p99 (us)", "AWS/SageMaker", "ModelLatency", "p99"),
    ("overhead latency p99 (us)", "AWS/SageMaker", "OverheadLatency", "p99"),
    ("cpu %", "/aws/sagemaker/Endpoints", "CPUUtilization", "Average"),
    ("memory %", "/aws/sagemaker/Endpoints", "MemoryUtilization", "Average"),
]

rows = []
for label, namespace, metric, statistic in queries:
    kwargs = dict(Namespace=namespace, MetricName=metric, Dimensions=dimensions,
                  StartTime=start, EndTime=end, Period=300)
    if statistic.startswith("p"):
        kwargs["ExtendedStatistics"] = [statistic]
    else:
        kwargs["Statistics"] = [statistic]
    points = cw.get_metric_statistics(**kwargs)["Datapoints"]
    values = [(p.get(statistic) if not statistic.startswith("p")
               else p["ExtendedStatistics"][statistic]) for p in points]
    rows.append({"metric": label, "datapoints": len(values),
                 "latest": round(values[-1], 2) if values else None,
                 "max": round(max(values), 2) if values else None})

display(pd.DataFrame(rows).set_index("metric"))
print("Latency is reported in MICROSECONDS. 25,000 here means 25ms, which is why "
      "the\np99 alarm threshold is 100,000 and not 100.")

In [ ]:
# Plot what the dashboard shows, so the figure can go in the design document
# and the video has something to point at even before the console loads.
import matplotlib.pyplot as plt

def series(namespace, metric, statistic, period=60):
    kwargs = dict(Namespace=namespace, MetricName=metric, Dimensions=dimensions,
                  StartTime=start, EndTime=end, Period=period)
    if statistic.startswith("p"):
        kwargs["ExtendedStatistics"] = [statistic]
    else:
        kwargs["Statistics"] = [statistic]
    points = sorted(cw.get_metric_statistics(**kwargs)["Datapoints"],
                    key=lambda p: p["Timestamp"])
    xs = [p["Timestamp"] for p in points]
    ys = [(p.get(statistic) if not statistic.startswith("p")
           else p["ExtendedStatistics"][statistic]) for p in points]
    return xs, ys

fig, axes = plt.subplots(1, 3, figsize=(15, 3.4))
x, y = series("AWS/SageMaker", "Invocations", "Sum")
axes[0].plot(x, y, color="#2f5d8a"); axes[0].set_title("Invocations (sum per minute)")

for statistic, color in (("p50", "#8aa6c1"), ("p99", "#c0392b")):
    x, y = series("AWS/SageMaker", "ModelLatency", statistic)
    axes[1].plot(x, [v / 1000 for v in y], color=color, label=f"model {statistic}")
axes[1].set_ylabel("milliseconds"); axes[1].set_title("Model latency"); axes[1].legend(fontsize=8)

for metric, color in (("CPUUtilization", "#2f5d8a"), ("MemoryUtilization", "#e67e22")):
    x, y = series("/aws/sagemaker/Endpoints", metric, "Average", period=300)
    axes[2].plot(x, y, color=color, label=metric)
axes[2].axhline(80, ls="--", lw=1, color="#c0392b", label="alarm 80%")
axes[2].set_ylim(0, 100); axes[2].set_ylabel("percent")
axes[2].set_title("Host utilisation"); axes[2].legend(fontsize=8)

for ax in axes:
    ax.tick_params(axis="x", rotation=30, labelsize=8)
plt.tight_layout()
path = os.path.join(FIG_DIR, "20_infrastructure_metrics.png")
plt.savefig(path, dpi=140, bbox_inches="tight")
store.upload_file(path, f"{C.PREFIX}/reports/figures/20_infrastructure_metrics.png")
plt.show()

## 5 · Prove an alarm actually fires

An untested alarm is a hypothesis. Rather than wait for a real incident, set
one alarm's state directly with `set_alarm_state` — this exercises the full
path including the SNS delivery, then the alarm re-evaluates against real data
on its next period.

In [ ]:
test_alarm = f"{ENDPOINT_NAME}-model-latency-p99"
cw.set_alarm_state(AlarmName=test_alarm, StateValue="ALARM",
                   StateReason="Synthetic test from notebook 09: verifying SNS delivery")
time.sleep(5)

state = cw.describe_alarms(AlarmNames=[test_alarm])["MetricAlarms"][0]
print(f"{test_alarm}\n  state  {state['StateValue']}\n  reason {state['StateReason']}")

history = cw.describe_alarm_history(AlarmName=test_alarm, HistoryItemType="StateUpdate",
                                    MaxRecords=5)["AlarmHistoryItems"]
display(pd.DataFrame([{"when": h["Timestamp"], "summary": h["HistorySummary"]}
                      for h in history]))
print("\nIf a confirmed email subscription exists, the notification has been sent.\n"
      "The alarm returns to OK on its next evaluation period against real metrics.")

## 6 · What the monitoring stack now covers

| Layer | Signal | Where it surfaces | Alarm |
|---|---|---|---|
| Infrastructure | invocations, 4xx/5xx, model and overhead latency, CPU/memory/disk | CloudWatch dashboard, bands 1–2 | 5 alarms → SNS |
| Data | PSI per numerical feature, missing rate, unseen categorical levels | hourly data-quality monitor (notebook 08) | violation count → CloudWatch |
| Model | log loss, AUC, calibration error against the approved baseline | hourly model-quality monitor (notebook 08) | constraint breach → CloudWatch → SNS |
| Pipeline | execution succeeded / failed | dashboard, bottom band | — |

**The gap worth naming:** the retraining trigger is designed but not wired in
this environment. The intent is an EventBridge rule on a breached
model-quality alarm starting a pipeline execution, with the candidate still
required to clear the quality gate and the manual approval gate. The Learner
Lab role cannot create that rule, so notebook 10 starts executions explicitly
and the automated trigger is recorded as a future enhancement rather than
claimed as working.

In [ ]:
record = {
    "created_utc": pd.Timestamp.utcnow().isoformat(),
    "region": region,
    "endpoint": ENDPOINT_NAME,
    "sns_topic": topic_arn,
    "dashboard": DASHBOARD_NAME,
    "dashboard_url": console,
    "alarms": [{"name": a["AlarmName"], "metric": a["MetricName"],
                "state": a["StateValue"],
                "threshold": a["Threshold"],
                "comparison": a["ComparisonOperator"]} for a in described],
    "metrics_observed": rows,
    "retraining_trigger": {
        "designed": "EventBridge rule on a breached model-quality alarm -> "
                    "StartPipelineExecution",
        "implemented": False,
        "reason": "Learner Lab execution role cannot create EventBridge rules; "
                  "executions are started explicitly in notebook 10",
    },
}
store.put_json(record, f"{C.ARTIFACTS_PREFIX}/infrastructure_monitoring.json")
with open(os.path.abspath("../artifacts/infrastructure_monitoring.json"), "w") as fh:
    json.dump(record, fh, indent=2, default=str)
print(json.dumps(record, indent=2, default=str)[:1100])